# 06e — Uncertainty Quantification

**Purpose:** Generate calibrated prediction intervals for Smooth + Erratic SKUs via conformal prediction. Intervals feed directly into safety stock in 06g.

**Inputs:**
- `tweedie_optimized_fold2.txt` + `tweedie_optimization_results.pkl` — 06d winner, and which experiment won
- `sku_regimes_fold2.parquet` — regime/routing from 06c
- `features_train_v2.parquet` / `feature_cols_v2.pkl`

**Outputs:**
- `conformal_residuals_fold2.pkl` — per-SKU residual distributions + locked service levels
- `coverage_validation_fold2.csv` — empirical vs. target coverage table

**Two fixes vs. the original plan spec, both handled in Section 1:**
1. **Residual space.** 06d converted the 7-day-target model's output to a "daily equivalent" and re-aggregated into calendar weeks — the rolling window and calendar week don't align, which distorts the metric. This notebook computes residuals directly in the winning model's native output space instead, so the mismatch never enters the calculation.
2. **Calibration vs. coverage-test split.** Calibrating intervals and validating their coverage on the same val data is circular. Fold 2 val is split chronologically so Section 2 checks coverage on genuinely held-out data — without touching Fold 3.

---

## Section 1 — Conformal Setup

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
PROCESSED_DIR    = '../data/processed'
CALIBRATION_DIR  = f'{PROCESSED_DIR}/calibration'
SEGMENTATION_DIR = f'{PROCESSED_DIR}/segmentation'
MODELS_DIR       = f'{PROCESSED_DIR}/models'

FOLD2_VAL_START     = '2014-02-01'
CALIBRATION_END     = '2014-08-31'   # ~7 months -> build residual distributions
COVERAGE_TEST_START = '2014-09-01'   # ~4 months -> held-out coverage check
FOLD2_VAL_END       = '2014-12-31'

# ── Load 06d winner + which experiment won ──────────────────────────────────
model = lgb.Booster(model_file=f'{MODELS_DIR}/tweedie_optimized_fold2.txt')

with open(f'{CALIBRATION_DIR}/tweedie_optimization_results.pkl', 'rb') as f:
    opt_results = pickle.load(f)
WINNER_MODEL = opt_results['winner']

# Which 06d experiments trained on the 7-day-forward-sum target vs next-day --
# determines what "native output space" means below. Read from 06d's own
# result rather than assumed, so this notebook doesn't silently break if a
# re-run picks a different winner.
SEVEN_DAY_TARGET_EXPERIMENTS = {'experiment_a', 'experiment_c'}
TARGET_IS_7DAY = WINNER_MODEL in SEVEN_DAY_TARGET_EXPERIMENTS
print(f'06d winner: {WINNER_MODEL}  |  native target: '
      f'{"7-day forward sum" if TARGET_IS_7DAY else "next-day"}')

# ── Load 06c routing ──────────────────────────────────────────────────────────
sku_regimes  = pd.read_parquet(f'{SEGMENTATION_DIR}/sku_regimes_fold2.parquet')
tweedie_skus = set(sku_regimes[sku_regimes['routing'] == 'tweedie']['id'].tolist())

# ── Load features, filter to Smooth+Erratic + full Fold 2 val window ────────
train_full = pd.read_parquet(f'{PROCESSED_DIR}/features/features_train_v2.parquet')
train_full['date'] = pd.to_datetime(train_full['date'])
with open(f'{PROCESSED_DIR}/features/feature_cols_v2.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

mask_se  = train_full['id'].isin(tweedie_skus)
mask_val = (train_full['date'] >= FOLD2_VAL_START) & (train_full['date'] <= FOLD2_VAL_END)
val_se = train_full[mask_val & mask_se].sort_values(['id', 'date']).copy()

# ── Build target + prediction in the model's NATIVE space ───────────────────
# No /7 conversion, no calendar-week re-aggregation -- this avoids 06d's
# alignment issue by never leaving the model's native units in the first place.
if TARGET_IS_7DAY:
    val_se['target'] = (
        val_se.groupby('id')['units_sold']
        .transform(lambda x: x.shift(-7).rolling(window=7, min_periods=7).sum())
    )
    val_se = val_se.dropna(subset=['target'])
    UNIT_LABEL = 'expected demand, rolling next-7-days'
else:
    val_se['target'] = val_se['units_sold']
    UNIT_LABEL = 'expected demand, next-day'

val_se['yhat']     = np.maximum(model.predict(val_se[feature_cols].values), 0)
val_se['residual'] = val_se['target'] - val_se['yhat']
print(f'Residual unit: {UNIT_LABEL}')
print(f'Rows: {len(val_se):,}  |  SKUs: {val_se["id"].nunique():,}')

# ── Calibration / coverage-test split (chronological, within Fold 2 val) ────
calib_df = val_se[val_se['date'] <= CALIBRATION_END]
test_df  = val_se[val_se['date'] >= COVERAGE_TEST_START]
print(f'Calibration rows: {len(calib_df):,}  |  Coverage-test rows: {len(test_df):,}')

# ── Per-SKU residual distributions -- built on calibration window ONLY ──────
sku_residuals = calib_df.groupby('id')['residual'].apply(list).to_dict()

missing = tweedie_skus - set(val_se['id'])
print(len(missing))
print(sku_regimes[sku_regimes['id'].isin(missing)]['routing'].value_counts())
# check if these SKUs simply have no rows at all in train_full, vs. got trimmed by dropna
print(train_full[train_full['id'].isin(missing)]['date'].agg(['min','max','count']))

pooled = pd.Series([r for v in sku_residuals.values() for r in v])
print(pooled.describe())
print('skew:', pooled.skew())

In [ ]:
in_window = train_full[train_full['id'].isin(missing) & mask_val]
print(f'Missing SKUs with rows actually in Fold 2 val window: {in_window["id"].nunique()} / 27')

print(calib_df.loc[calib_df['residual'].idxmin(), ['id','date','target','yhat','residual']])
print(calib_df.loc[calib_df['residual'].idxmax(), ['id','date','target','yhat','residual']])

# 1. Regime classification
print(sku_regimes[sku_regimes['id'] == 'HOUSEHOLD_1_474_TX_2_validation'])

# 2. The actual spike window (Feb 10–25, 2014) — not head/tail
spike_window = val_se[
    (val_se['id'] == 'HOUSEHOLD_1_474_TX_2_validation') &
    (val_se['date'] >= '2014-02-10') & (val_se['date'] <= '2014-02-25')
]
print(spike_window[['date', 'units_sold']].to_string(index=False))


### Section 1 Findings — Conformal Setup

- Winning model / native target space: _[fill in from print output]_
- SKUs with residual distributions: _[fill in]_
- Residual shape (symmetric / skewed / outliers) on calibration window: _[fill in]_

---

## Section 2 — Coverage Validation

**Hard gate.** Empirical coverage on the *held-out coverage-test window* (never used for calibration) must land within ±3pp of target at each service level. Fail here → diagnose whether it's global or concentrated in one regime before moving to Section 3 — don't build on a broken calibration.

In [ ]:
SERVICE_LEVELS        = [0.50, 0.75, 0.80, 0.90, 0.95, 0.99]
COVERAGE_TOLERANCE_PP = 3.0
MIN_RESIDUALS_PER_SKU = 5   # skip SKUs with too few calibration points for a stable quantile

def empirical_coverage(test_df, sku_residuals, level):
    covered, total = 0, 0
    for sku_id, grp in test_df.groupby('id'):
        residuals = sku_residuals.get(sku_id, [])
        if len(residuals) < MIN_RESIDUALS_PER_SKU:
            continue
        upper_bound = grp['yhat'] + np.percentile(residuals, level * 100)
        covered += (grp['target'] <= upper_bound).sum()
        total   += len(grp)
    return covered / total if total else np.nan

coverage_rows = []
for level in SERVICE_LEVELS:
    emp = empirical_coverage(test_df, sku_residuals, level)
    target_pct, emp_pct = level * 100, emp * 100
    coverage_rows.append({
        'service_level': f'q{int(target_pct)}',
        'target_pct':    target_pct,
        'empirical_pct': round(emp_pct, 1),
        'pass':          abs(emp_pct - target_pct) <= COVERAGE_TOLERANCE_PP,
    })

coverage_df = pd.DataFrame(coverage_rows)
print(coverage_df.to_string(index=False))

coverage_df.to_csv(f'{CALIBRATION_DIR}/coverage_validation_fold2.csv', index=False)

if not coverage_df['pass'].all():
    print('\nWARNING: coverage gate FAILED at one or more levels. '
          'Check whether the miss is global or concentrated in one regime '
          '(split test_df by sku_regimes before proceeding).')

### Section 2 Findings — Coverage Validation

_[paste the printed table here after running]_

**Gate status:** _[PASS/FAIL]_

---

## Section 3 — Per-SKU Interval Width Analysis

Confirm intervals behave sensibly: wider for sparse/uncertain SKUs, roughly proportional to demand scale. Flag SKUs where q90 width is extreme relative to their own demand — candidates for a second look in 06f.

In [ ]:
import matplotlib.pyplot as plt

WIDTH_FLAG_MULTIPLE = 3.0  # flag if q90 width > 3x mean demand

sku_summary = (
    calib_df.groupby('id')
    .agg(mean_demand=('target', 'mean'), zero_rate=('units_sold', lambda x: (x == 0).mean()))
    .reset_index()
)
sku_summary['q90_width'] = sku_summary['id'].map(
    lambda sid: np.percentile(sku_residuals[sid], 90)
    if len(sku_residuals.get(sid, [])) >= MIN_RESIDUALS_PER_SKU else np.nan
)

print(f"Median q90 interval width: {sku_summary['q90_width'].median():.2f}")

flagged = sku_summary[sku_summary['q90_width'] > WIDTH_FLAG_MULTIPLE * sku_summary['mean_demand']]
print(f'SKUs flagged for extreme interval width: {len(flagged)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(sku_summary['q90_width'].dropna(), bins=40)
axes[0].set_title('q90 interval width distribution')

axes[1].scatter(sku_summary['zero_rate'], sku_summary['q90_width'], alpha=0.3, s=8)
axes[1].set_xlabel('zero rate'); axes[1].set_ylabel('q90 width')
axes[1].set_title('Width vs. sparsity')

axes[2].scatter(sku_summary['mean_demand'], sku_summary['q90_width'], alpha=0.3, s=8)
axes[2].set_xlabel('mean demand'); axes[2].set_ylabel('q90 width')
axes[2].set_title('Width vs. demand scale')

plt.tight_layout()
plt.show()

### Section 3 Findings — Interval Width Analysis

- Median q90 interval width: _[fill in]_
- Widths scale with sparsity/demand as expected: _[yes/no — explain any anomaly]_
- SKUs flagged for extreme width: _[count — borderline Intermittent candidates for 06f]_

---

## Section 4 — Service Level Selection by Regime

**Why not just pick q80/q90 by convention:** service level should reflect the actual tradeoff between stockout cost and holding cost, and Erratic SKUs' higher demand variance (CV² ≥ 0.49, by the 06c classification) changes that tradeoff versus Smooth SKUs even under the same cost assumptions. Two layers:

**Layer 1 — Newsvendor critical ratio.** The cost-minimizing service level is `Cu / (Cu + Co)`, where `Cu` = stockout cost per unit short, `Co` = holding cost per unit excess. No per-SKU financial data exists in this dataset, so this uses a commonly cited retail assumption — stockout cost runs 3–5x holding cost — as a documented, adjustable parameter rather than an unstated one.

**Layer 2 — Empirical cross-check.** Within that range, use Section 3's width-vs-level relationship per regime to confirm the chosen level sits at or before the point where interval width starts growing faster than coverage gained — i.e. not paying for safety stock a level buys little of.

In [ ]:
# ── Layer 1: newsvendor critical ratio (economic baseline) ─────────────────
COST_RATIO_LOW, COST_RATIO_HIGH = 3.0, 5.0   # stockout cost as a multiple of holding cost
critical_ratio_low  = COST_RATIO_LOW  / (COST_RATIO_LOW  + 1)
critical_ratio_high = COST_RATIO_HIGH / (COST_RATIO_HIGH + 1)
print(f'Critical ratio range: {critical_ratio_low:.2f}-{critical_ratio_high:.2f}')

# Erratic SKUs (CV² >= 0.49) carry more demand variance than Smooth SKUs. At
# the same cost ratio, a fixed quantile buys less absolute stockout
# protection against a spike for a high-variance SKU -- so Erratic is placed
# at the top of the critical-ratio range, Smooth at the bottom, rather than
# using one flat level for both regimes.
CANDIDATE_SERVICE_LEVEL = {
    'smooth':  round(critical_ratio_low, 2),
    'erratic': round(critical_ratio_high, 2),
}

def nearest_level(x):
    return min(SERVICE_LEVELS, key=lambda l: abs(l - x))

DEFAULT_SERVICE_LEVEL = {regime: nearest_level(v) for regime, v in CANDIDATE_SERVICE_LEVEL.items()}
print(f'Candidate service levels (economic baseline): {CANDIDATE_SERVICE_LEVEL}')
print(f'Snapped to evaluated levels: {DEFAULT_SERVICE_LEVEL}')

# ── Layer 2: empirical cross-check against Section 3's width curve ─────────
# TODO: for each regime, compute median interval width at each SERVICE_LEVELS
#       step; confirm DEFAULT_SERVICE_LEVEL sits before the width-vs-level
#       curve steepens noticeably (diminishing returns). If it doesn't,
#       document why the economic baseline is being overridden.

print(f'\nFinal locked service levels: {DEFAULT_SERVICE_LEVEL}')

### Section 4 Findings — Service Level Selection

- Economic baseline (critical ratio): _[fill in from print output]_
- Empirical cross-check result: _[does the chosen level sit before the width curve steepens? fill in]_
- Final locked levels: _[fill in — confirm q80/q90, or note deviation and why]_

**Scope note for 06g:** the winning model's native unit may be a rolling 7-day-forward sum, not a fixed Mon–Sun calendar week. 06g's reorder simulation should evaluate on the same cadence this notebook used (e.g. check inventory and next-7-day expected demand every 7 days from a fixed anchor date) rather than assuming calendar-week boundaries — flagged here so it isn't silently mismatched again.

In [ ]:
with open(f'{CALIBRATION_DIR}/conformal_residuals_fold2.pkl', 'wb') as f:
    pickle.dump({
        'sku_residuals':         sku_residuals,
        'default_service_level': DEFAULT_SERVICE_LEVEL,
        'residual_unit':         UNIT_LABEL,
        'winner_model':          WINNER_MODEL,
        'cost_ratio_assumption': (COST_RATIO_LOW, COST_RATIO_HIGH),
    }, f)

print('06e complete. Proceed to 06f_intermittent_demand.ipynb.')